In [ ]:
import json
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
import re
from tqdm import tqdm

In [ ]:
with open("./datasets/HateNorm/train.json", "r") as file:
    train_json = json.load(file)
with open("./datasets/HateNorm/valid.json", "r") as file:
    valid_json = json.load(file)
with open("./datasets/HateNorm/test.json", "r") as file:
    test_json = json.load(file)

In [ ]:
model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [ ]:
data_train_list = []
data_train_list_4train = []
data_valid_list = []
data_valid_list_4train = []
data_test_list = []
system = """
        You are a strict hate speech detector.
        Hate speech is defined as the public expression of prejudice, hostility, or offensive remarks directed towards specific groups or individuals based on their identity characteristics, such as race, ethnicity, gender, or religious beliefs.
        The input is a list of words from the sentence. Your task is to assign each word a tag of either 1 or 0. Assign 1 to words that appear to be hate speech, and 0 to all other words.
        Format the result in JSON format, with the key “result” and the value as a list of dictionaries.
        Output should be in the following JSON format: {"result": [{"text": word1, "label": binary tag}, {"text": word2, "label": binary tag}, ...]}
"""

for data in train_json:
    target_sentence = data['sentence_tokens']
    user = f"The number of words in the input sentence is {len(target_sentence)}." + "\n" + f"input: {target_sentence}" + "\n"
    completion_list = [{"text": word, "label": tag} for word,tag in zip(data['sentence_tokens'], data['labels'])]
    completion = "{"+f'"result": {completion_list}'+"}"
    messages = [
        {"role":"system", "content":system},
        {"role":"user", "content":user},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    data_train_list_4train.append({"prompt":prompt, "completion":completion})
    data_dict = {"prompt":prompt, "completion":completion}
    data_dict.update(data)
    data_train_list.append(data_dict)

for data in valid_json:
    target_sentence = data['sentence_tokens']
    user = f"The number of words in the input sentence is {len(target_sentence)}." + "\n" + f"input: {target_sentence}" + "\n"
    completion_list = [{"text": word, "label": tag} for word,tag in zip(data['sentence_tokens'], data['labels'])]
    completion = "{"+f'"result": {completion_list}'+"}"
    messages = [
        {"role":"system", "content":system},
        {"role":"user", "content":user}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    data_dict = {"prompt":prompt, "completion":completion}
    data_dict_4train = {"prompt":prompt, "completion":completion}
    data_dict.update(data)
    data_valid_list.append(data_dict)
    data_valid_list_4train.append(data_dict_4train)

for data in test_json:
    target_sentence = data['sentence_tokens']
    user = f"The number of words in the input sentence is {len(target_sentence)}." + "\n" + f"input: {target_sentence}" + "\n"
    completion_list = [{"text": word, "label": tag} for word,tag in zip(data['sentence_tokens'], data['labels'])]
    completion = "{"+f'"result": {completion_list}'+"}"
    messages = [
        {"role":"system", "content":system},
        {"role":"user", "content":user}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    data_dict = {"prompt":prompt, "completion":completion}
    data_dict.update(data)
    data_test_list.append(data_dict)
    

In [ ]:
dataset_train = Dataset.from_list(data_train_list_4train)
dataset_valid = Dataset.from_list(data_valid_list_4train)
dataset_test = Dataset.from_list(data_test_list)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# LoRA の設定（peft_config）
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# SFT のハイパーパラメータ（Arguments 相当）
sft_config = SFTConfig(
    output_dir="./results/Qwen/",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    learning_rate=5e-5,
    max_length=2048,
    logging_strategy="epoch",
    logging_dir="./logs",
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=1,
    bf16=True,
    packing=False,
    completion_only_loss=True,
    load_best_model_at_end=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_train,
    eval_dataset=dataset_valid,
    processing_class=tokenizer,
    args=sft_config,
    peft_config=peft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()
trainer.save_model()

In [ ]:
data_pred = []
for data in tqdm(data_test_list[606:]):
    chat_prompt = data['prompt']
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated = output_text.strip()
    
    matched_list = re.findall(r'\{\"result\": \[[^\]]*\]\}' ,generated)
    pred_words = []
    pred_labels = []
    if len(matched_list)<=1:
        error_message = "format error"
        acc = 0
    else:
        matched_text = matched_list[1].replace("}.", "},").replace("\\\'", "\'").replace("\'\"\'", "\'\\\"\'").replace("\'text\'", "\"text\"").replace("\'label\'", "\"label\"")
        matched_text = re.sub("\":\s?\'", r'": "', matched_text)
        matched_text = re.sub("\'[,.]\s?\"label", r'", "label', matched_text)
        matched_text = re.sub("},\s?\'([^{}[]]*\",\s?\"label\")", r'}, {"text": "\1', matched_text)
        matched_text = re.sub("\\\([^u\"])", r'\1', matched_text)
        pred_dict = json.loads(matched_text)
        pred_result = pred_dict['result']
        if len(pred_result)!=len(data['labels']):
            error_message = "labels length error"
            acc = 0
        else:
            error_message = "no error"
            for pred,word in zip(pred_result, data['sentence_tokens']):
                pred_labels.append(pred['label'])
                if pred['label']==1:
                    pred_words.append(word)
            acc = sum([int(a==b) for a,b in zip(data['labels'], pred_labels)])/len(pred_result)
    data_pred.append({
        'ID':data['Id'], 
        'true':data['labels'], 
        'pred':pred_labels, 
        'words':pred_words, 
        'accuracy':acc, 
        'usage':data['usage'], 
        'labels_bio':data['labels_bio'], 
        'error_message':error_message,
        'generated_text':generated})

for i in range(len(data_pred)):
    if data_pred[i]['error_message']!="no error":
        data_pred[i]['is_error'] = True
    else:
        pred_bio = []
        pre_p = 0
        for p in data_pred[i]['pred']:
            if p==0:
                pred_bio.append('O')
            elif p==1:
                if pre_p == 0:
                    pred_bio.append('B')
                else:
                    pred_bio.append('I')
            pre_p = p
        data_pred[i]['pred_bio'] = pred_bio
        if len(pred_bio)!=len(data_pred[i]['labels_bio']):
            data_pred[i]['is_error'] = True
        else:
            data_pred[i]['is_error'] = False

In [ ]:
#binary F1
cm_bi = [[0,0],[0,0]]
for data in data_pred:
    if not data['is_error']:
        for p,t in zip(data['pred'],data['true']):
            cm_bi[p][t] += 1
precision = cm_bi[1][1]/(cm_bi[1][1]+cm_bi[1][0])
recall = cm_bi[1][1]/(cm_bi[1][1]+cm_bi[0][1])
f1_bi = 2*precision*recall/(precision+recall)
print(f"binary F1: {f1_bi:.5f}")
print(f"binary precision: {precision:.5f}")
print(f"binary recall: {recall:.5f}")
#soft F1
cm_bio = [[0,0,0],[0,0,0],[0,0,0]]
bio2num = {'B':1,'I':2,'O':0}
for data in data_pred:
    if not data['is_error']:
        for p,t in zip(data['pred_bio'],data['labels_bio']):
            cm_bio[bio2num[p]][bio2num[t]] += 1
pre = (cm_bio[1][1]+cm_bio[2][2])/(sum(cm_bio[1])+sum(cm_bio[2]))
rec = (cm_bio[1][1]+cm_bio[2][2])/(cm_bio[0][1]+cm_bio[1][1]+cm_bio[2][1]+cm_bio[0][2]+cm_bio[1][2]+cm_bio[2][2])
f1_bio = 2*pre*rec/(pre+rec)
print(f"soft F1: {f1_bio:.5f}")
print(f"soft precision: {pre:.5f}")
print(f"soft recall: {rec:.5f}")
#hard F1
y_true = []
y_pred = []
for data in data_pred:
    if not data['is_error']:
        y_true.append(data['labels_bio'])
        y_pred.append(data['pred_bio'])
f1_span = f1_score(y_true, y_pred, average='macro')
pre_span = precision_score(y_true, y_pred, average='macro')
rec_span = recall_score(y_true, y_pred, average='macro')
print(f"hard F1: {f1_span:.5f}")
print(f"hard pre: {pre_span:.5f}")
print(f"hard rec: {rec_span:.5f}")